In [0]:
%sql
-- qdp_lakebase.public
USE CATALOG IDENTIFIER(:p_catalog_name);

CREATE SCHEMA IF NOT EXISTS qdp_locations_all;


In [0]:
-- Create Quantexa Data Product organisations_all Schema metadata
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);


-- metadata table
DROP TABLE IF EXISTS metadata;

CREATE OR REPLACE TABLE metadata_table (
	data_product_version VARCHAR,
	schema_name VARCHAR,
	schema_version VARCHAR,
	schema_variant_name VARCHAR,
	schema_variant_version VARCHAR,
	instance VARCHAR,
	instance_version VARCHAR
 );


INSERT INTO metadata 
(
  supplier, 
  data_product,
  data_product_version,
  schema_name,
  schema_version,
  schema_variant_name,
  schema_variant_version,
  instance,
  instance_version
) 
VALUES (
  'quantexa.com',
  'com.quantexa.qdp.locations_all',
  '0.1',
  'locations_all.data_vault',
  '0.1',
  'base',
  '0.1',
  'not yet populated with data',
  ''
);



In [0]:
-- Create Quantexa Data Product locations_all Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

-- Address  tables
DROP TABLE IF EXISTS hub_address;
DROP TABLE IF EXISTS sat_address;


CREATE TABLE hub_address ( 
	address_id BIGINT NOT NULL GENERATED  ALWAYS AS IDENTITY  (START WITH 0 INCREMENT BY 1) COMMENT 'Address Identity',
	address_entity_id STRING,
	tennant_id BIGINT NOT NULL DEFAULT 0,
	period_start TIMESTAMP,
	period_end TIMESTAMP,
	CONSTRAINT pk_hub_address_addressid PRIMARY KEY ( address_id )
 )     USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

CREATE TABLE sat_address ( 
	sat_address_id BIGINT NOT NULL COMMENT 'Address Identity Sat',
	address_id BIGINT NOT NULL,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	postal_code STRING,
	data_source_code STRING,
	data_source_concept_id BIGINT,
	data_source_raw_code STRING,
	data_source_raw_concept_id BIGINT,
	type_code STRING,
	type_concept_id BIGINT,
	type_raw_code STRING,
	type_raw_concept_id BIGINT,
	flatnumber_housenumber_housename STRING,
	house_number STRING,
	house_name STRING,
	building_number STRING,
	building_name STRING,
	address_line1 STRING,
	address_line2 STRING,
	address_line3 STRING,
	address_line4 STRING,
	address_line5 STRING,
	address_line6 STRING,
	address_line7 STRING,
	address_line8 STRING,
	street STRING,
	district STRING,
	city STRING,
	county STRING,
	state STRING,
	country STRING,
	country_code STRING,
	country_concept_id BIGINT,
	country_raw_code STRING,
	country_raw_concept_id BIGINT,
	full_address STRING,
	uprn STRING,
	what3words STRING,
	latitude DOUBLE,
	longitude DOUBLE,
	directions STRING,
	period_start TIMESTAMP,
	period_end TIMESTAMP,
	CONSTRAINT pk_locationall_sataddress_sataddressid PRIMARY KEY ( sat_address_id)
 )     USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

DROP TABLE IF EXISTS sat_physical_location;
CREATE TABLE sat_physical_location ( 
	physical_location_id BIGINT NOT NULL GENERATED  ALWAYS AS IDENTITY  (START WITH 0 INCREMENT BY 1) COMMENT 'Physical Location Identity',
	address_id BIGINT,
	load_datetime TIMESTAMP NOT NULL DEFAULT CURRENT_TIMESTAMP,
	record_source_id BIGINT NOT NULL DEFAULT 1,
	location_name STRING,
	data_source_code STRING,
	data_source_concept_id BIGINT,
	data_source_raw_code STRING,
	data_source_raw_concept_id BIGINT,
	type_code STRING,
	type_concept_id BIGINT,
	type_raw_code STRING,
	type_raw_concept_id BIGINT,
	latitude DOUBLE,
	longitude DOUBLE,
	altitude STRING,
	altitude_mean_sea_level STRING,
	horizontal_accuracy STRING,
	vertical_accuracy STRING,
	what3words STRING,
	city STRING,
	county STRING,
	state STRING,
	country STRING,
	country_code STRING,
	country_concept_id BIGINT,
	country_raw_code STRING,
	country_raw_concept_id BIGINT,
	directions STRING,
	is_on_land BOOLEAN,
	is_on_water BOOLEAN,
	is_at_sea BOOLEAN,
	period_start TIMESTAMP,
	period_end TIMESTAMP,
	CONSTRAINT pk_sat_physical_location PRIMARY KEY ( physical_location_id )
 )     USING DELTA
TBLPROPERTIES ('delta.feature.allowColumnDefaults' = 'supported');

ALTER TABLE sat_address ADD CONSTRAINT fk_sat_address_hub_address FOREIGN KEY ( address_id ) REFERENCES hub_address( address_id ) ON DELETE NO ACTION ON UPDATE NO ACTION;

ALTER TABLE sat_physical_location ADD CONSTRAINT fk_sat_physical_location_hub_address FOREIGN KEY ( address_id ) REFERENCES hub_address( address_id ) ON DELETE NO ACTION ON UPDATE NO ACTION;

ALTER TABLE hub_address ALTER COLUMN address_id comment 'Address Identity';

ALTER TABLE sat_address ALTER COLUMN sat_address_id comment 'Address Identity Sat';

ALTER TABLE sat_physical_location ALTER COLUMN physical_location_id comment 'Physical Location Identity';



In [0]:
-- Create a default NO KNOWN ADDRESS record in the Data Product locations_all Schema
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

INSERT INTO hub_address ( 
	address_entity_id,
	period_start,
	period_end
) VALUES (
  NULL,
  NULL,
  NULL
)
;

INSERT INTO sat_address ( 
	sat_address_id,
	address_id,
	postal_code,
	address_line1,
	full_address,
	period_start,
	period_end
  ) VALUES (
  0,
  0,
  NULL,
  "NO KNOWN ADDRESS",
  "NO KNOWN ADDRESS",
  NULL,
  NULL
 )
 ;


In [0]:
USE CATALOG IDENTIFIER(:p_catalog_name);
USE SCHEMA IDENTIFIER(:p_schema_name);

CREATE OR REPLACE VIEW view_addesses AS 
SELECT 
  h.tennant_id,
  t.tennant_name,
  h.address_id,
  h.address_entity_id,
  s.postal_code,
  s.type_code,
  s.house_number,
  s.house_name,
  s.building_number,
  s.building_name,
  s.address_line1,
  s.address_line2,
  s.address_line3,
  s.address_line4,
  s.address_line5,
  s.address_line6,
  s.address_line7,
  s.address_line8,
  s.street,
  s.district,
  s.city,
  s.county,
  s.state,
  s.country,
  s.country_code,
  s.country_concept_id,
  s.country_raw_code,
  s.country_raw_concept_id,
  s.full_address,
  s.uprn,
  s.what3words,
  s.latitude,
  s.longitude,
  s.directions,
  h.period_start,
  h.period_end, 
  s.address_id as sat_address_hub_key,
  s.sat_address_id,
  s.type_concept_id,
  s.type_raw_code,
  s.type_raw_concept_id,
  s.period_start AS sat_address_period_start,
  s.period_end AS sat_address_period_end,
  s.load_datetime,
  s.record_source_id,
  s.data_source_code,
  s.data_source_concept_id,
  s.data_source_raw_code,
  s.data_source_raw_concept_id

    FROM demo_banking_silver.qdp_locations_all.hub_address h
    JOIN demo_banking_silver.qdp_locations_all.sat_address s
      ON h.address_id = s.address_id
    JOIN qdp_refcode_silver.qdp_reference_codes.tennant t
      on h.tennant_id = t.tennant_id
;



SELECT * FROM view_addesses;
